In [3]:
import math, random, os
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_moons

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", device)

PyTorch: 2.9.0+cpu | Device: cpu


# 6) Encoder–Decoder (Seq2Seq) — Đảo chuỗi (toy)

In [4]:

import string
alphabet = string.ascii_lowercase + " "
vocab = sorted(set(alphabet))
stoi = {ch:i+1 for i,ch in enumerate(vocab)}; itos = {i:s for s,i in stoi.items()}
PAD=0; max_len=16

def enc(s):
    s=s.lower()[:max_len]
    ids=[stoi.get(ch,0) for ch in s]; ids += [PAD]*(max_len-len(ids))
    return ids
def dec(ids): return "".join(itos.get(i,"?") for i in ids if i!=PAD)

N=2000; rng=np.random.default_rng(0)
X=[]; Y=[]
for _ in range(N):
    L=rng.integers(4,max_len+1)
    s="".join(rng.choice(list(vocab), size=L)); t=s[::-1]
    X.append(enc(s)); Y.append(enc(t))
X=torch.tensor(X); Y=torch.tensor(Y)
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.2, random_state=42)

class Seq2Seq(nn.Module):
    def __init__(self, V, d=128):
        super().__init__()
        self.emb=nn.Embedding(V+1, d, padding_idx=PAD)
        self.enc=nn.LSTM(d,d,batch_first=True)
        self.dec=nn.LSTM(d,d,batch_first=True)
        self.fc=nn.Linear(d,V+1)
    def forward(self, src, tgt):
        _,(h,c)=self.enc(self.emb(src))
        out,_=self.dec(self.emb(tgt), (h,c))
        return self.fc(out)

model=Seq2Seq(len(vocab)).to(device)
opt=optim.Adam(model.parameters(), lr=1e-3)
crit=nn.CrossEntropyLoss(ignore_index=PAD)
B=64
loader=DataLoader(TensorDataset(Xtr,Ytr), batch_size=B, shuffle=True)

for ep in range(1, 9):
    model.train(); loss_sum=0.0
    for xb,yb in loader:
        xb,yb=xb.to(device), yb.to(device)
        inp=torch.roll(yb,1,1); inp[:,0]=PAD
        opt.zero_grad()
        logits=model(xb, inp)
        loss=crit(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        loss.backward(); opt.step()
        loss_sum += loss.item()*xb.size(0)
    print(f"Epoch {ep:02d}: train_loss={loss_sum/len(Xtr):.3f}")

@torch.no_grad()
def infer(s):
    src=torch.tensor([enc(s)], dtype=torch.long).to(device)
    _,(h,c)=model.enc(model.emb(src))
    prev=torch.tensor([[PAD]], dtype=torch.long).to(device)
    out=[]
    for _ in range(max_len):
        dec_out,(h,c)=model.dec(model.emb(prev), (h,c))
        nxt=model.fc(dec_out[:,-1,:]).argmax(-1)
        out.append(nxt.item()); prev=nxt.unsqueeze(1)
    return dec(out)

for s in ["hello world", "abc xyz", "vietnam", "data science"]:
    print(s, "->", infer(s))


Epoch 01: train_loss=3.321
Epoch 02: train_loss=3.200
Epoch 03: train_loss=2.839
Epoch 04: train_loss=2.453
Epoch 05: train_loss=2.060
Epoch 06: train_loss=1.723
Epoch 07: train_loss=1.492
Epoch 08: train_loss=1.333
hello world -> ldowr leodlqihob
abc xyz -> zyxc badztnioufr
vietnam -> manites egvorbdz
data science -> ecesinac dthlxan
